# Classification Track — Parts A & B — Loan Status Prediction
**23CSE301 Machine Learning Capstone — Reviews 1 & 2**

Pipeline: data loading → EDA → cleaning & feature engineering → 5 Part-A classification
algorithms (Logistic Regression, KNN, Naive Bayes, Decision Tree, SVM) → evaluation
(Accuracy, Precision, Recall, F1-weighted, Confusion Matrix) → preliminary comparison table.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.svm import SVC
import joblib, os

from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              confusion_matrix, ConfusionMatrixDisplay, classification_report)

RANDOM_STATE = 42
sns.set_style('whitegrid')
sns.set_palette('colorblind')
plt.rcParams['figure.figsize'] = (8, 5)

## Section A — Dataset Loading & EDA

### A1. Dataset loading & audit

In [ ]:
DATA_PATH = "../data/Loan_approval_data_2025.csv"
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
df.head()


In [ ]:
df.info()


In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing = missing[missing > 0]
print("Columns with missing values:")
print(missing if len(missing) else "None found.")


In [ ]:
target = 'loan_status'
print(df[target].value_counts())
print()
print(df[target].value_counts(normalize=True).round(3))


In [ ]:
ax = df[target].value_counts().plot(kind='bar', color=sns.color_palette('colorblind'))
plt.title('Target Distribution: loan_status')
plt.xlabel('loan_status'); plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


### A2. EDA Visualisations

In [ ]:
num_cols = ['age', 'years_employed', 'annual_income', 'credit_score', 'credit_history_years',
            'savings_assets', 'current_debt', 'delinquencies_last_2yrs', 'derogatory_marks',
            'loan_amount', 'interest_rate', 'debt_to_income_ratio', 'loan_to_income_ratio',
            'payment_to_income_ratio']
num_cols = [c for c in num_cols if c in df.columns]

fig, axes = plt.subplots(4, 4, figsize=(18, 14))
axes = axes.flatten()
for i, c in enumerate(num_cols):
    sns.histplot(df[c].dropna(), kde=True, ax=axes[i])
    axes[i].set_title(f'Distribution of {c}')
for j in range(len(num_cols), len(axes)):
    fig.delaxes(axes[j])
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(11,9))
sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', square=True,
            annot_kws={'size': 7})
plt.title('Correlation Heatmap — Numeric Features')
plt.tight_layout()
plt.show()


> ### 📊 Key Observations: Numeric Feature Correlation Analysis (Loan Dataset)
> * **Perfect Collinearity & Feature Redundancy:** A near-perfect correlation ($r = 1.00$) exists between `loan_to_income_ratio` and `payment_to_income_ratio`, along with strong correlation to `loan_amount` ($r = 0.56$). Retaining both identical ratio metrics introduces strict collinear redundancy, which can distort standard error estimates in linear classifiers like Logistic Regression.
> * **Financial Capacity Coupling:** `annual_income` exhibits a strong positive correlation with `current_debt` ($r = 0.70$) and `loan_amount` ($r = 0.51$). Higher-earning applicants naturally secure larger lines of credit and carry larger nominal balances, while their leverage is captured via `debt_to_income_ratio` ($r = 0.60$ with `current_debt`).
> * **Demographic & Credit Tenure Alignment:** Expected life-cycle correlations appear across `age`, `years_employed` ($r = 0.53$), and `credit_history_years` ($r = 0.54$), reflecting natural career progression and credit file maturation over time.
> * **Risk Signals & Negative Associations:** `credit_score` exhibits notable inverse correlations with `interest_rate` ($r = -0.49$), `delinquencies_last_2yrs` ($r = -0.28$), and `derogatory_marks` ($r = -0.27$), accurately capturing risk-based loan pricing mechanics where riskier credit profiles incur higher borrowing costs.
> * **Modeling Implication:** The combination of non-linear risk boundaries (credit score thresholds) and isolated clusters of collinear debt ratios favors non-parametric tree ensembles (Gradient Boosting / Random Forest), which remain invariant to monotonic feature transformations and isolate redundant variables through orthogonal feature splitting.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13,5))
sns.scatterplot(data=df, x='annual_income', y='loan_amount', hue=target, alpha=0.5, ax=axes[0])
axes[0].set_title('Loan Amount vs Annual Income, by loan_status')

sns.scatterplot(data=df, x='credit_score', y='debt_to_income_ratio', hue=target, alpha=0.5, ax=axes[1])
axes[1].set_title('DTI Ratio vs Credit Score, by loan_status')
plt.tight_layout()
plt.show()


## Section B — Preprocessing & Feature Engineering

### B1. Data cleaning

In [ ]:
dupes = df.duplicated().sum()
print(f"Duplicate rows: {dupes}")
df = df.drop_duplicates()

# Outlier check via IQR on key skewed numeric columns
for col in ['annual_income', 'loan_amount', 'current_debt']:
    if col in df.columns:
        q1, q3 = df[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
        n_out = ((df[col] < lower) | (df[col] > upper)).sum()
        print(f"{col}: {n_out} potential outliers outside [{lower:.0f}, {upper:.0f}]")

# Strategy: cap at 1st/99th percentile instead of dropping rows (avoid losing genuine
# high-income / high-loan-amount applicants, which are meaningful, not noise).
for col in ['annual_income', 'loan_amount', 'current_debt']:
    if col in df.columns:
        low, high = df[col].quantile([0.01, 0.99])
        df[col] = df[col].clip(low, high)


In [ ]:
drop_cols = [c for c in ['customer_id'] if c in df.columns]
df = df.drop(columns=drop_cols)
print("Dropped identifier columns:", drop_cols)


### B2. Encoding, scaling & splitting

In [ ]:
y_raw = df[target]
X = df.drop(columns=[target])

categorical_cols = [c for c in ['occupation_status', 'product_type', 'loan_intent',
                                 'defaults_on_file'] if c in X.columns]
numeric_cols = [c for c in X.columns if c not in categorical_cols]

print("Numeric:", numeric_cols)
print("Categorical:", categorical_cols)

# Encode target if it's text (e.g. 'Approved'/'Denied' or 'Default'/'Fully Paid')
if y_raw.dtype == object:
    classes = sorted(y_raw.unique())
    class_map = {c: i for i, c in enumerate(classes)}
    y = y_raw.map(class_map)
    print("Target classes:", class_map)
else:
    y = y_raw

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(X_train.shape, X_test.shape)


In [ ]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_cols),
    ('cat', categorical_transformer, categorical_cols)
])


### B3. Feature engineering

In [ ]:
# Engineered feature: disposable_income_ratio — how much income remains after
# covering current debt, as a fraction of annual income. This can capture repayment
# capacity in a way no single existing ratio column does directly.
for split_df in (X_train, X_test):
    split_df['disposable_income_ratio'] = (
        (split_df['annual_income'] - split_df['current_debt']) / split_df['annual_income']
    )
numeric_cols = numeric_cols + ['disposable_income_ratio']

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_cols),
    ('cat', categorical_transformer, categorical_cols)
])


## Section D — Classification Track, Part A: 5 Algorithms

In [ ]:
results = {}

def evaluate(name, pipeline, X_tr, y_tr, X_te, y_te):
    pipeline.fit(X_tr, y_tr)
    preds = pipeline.predict(X_te)
    acc = accuracy_score(y_te, preds)
    prec = precision_score(y_te, preds, average='weighted', zero_division=0)
    rec = recall_score(y_te, preds, average='weighted', zero_division=0)
    f1 = f1_score(y_te, preds, average='weighted', zero_division=0)
    results[name] = {'model': pipeline, 'Accuracy': acc, 'Precision': prec,
                      'Recall': rec, 'F1_weighted': f1, 'preds': preds}
    print(f"{name:28s} Acc={acc:.4f}  Prec={prec:.4f}  Rec={rec:.4f}  F1={f1:.4f}")
    return pipeline


In [ ]:
# 1. Logistic Regression (baseline)
log_pipe = Pipeline([('prep', preprocessor), ('model', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))])
evaluate('Logistic Regression', log_pipe, X_train, y_train, X_test, y_test)

# Interpret coefficients / odds ratios
log_model = results['Logistic Regression']['model'].named_steps['model']
feature_names = results['Logistic Regression']['model'].named_steps['prep'].get_feature_names_out()
if log_model.coef_.shape[0] == 1:
    coef_df = pd.DataFrame({'feature': feature_names, 'coefficient': log_model.coef_[0]})
    coef_df['odds_ratio'] = np.exp(coef_df['coefficient'])
    print(coef_df.sort_values('coefficient', key=abs, ascending=False).head(10))


In [ ]:
# 2. K-Nearest Neighbors (tune k, note: distance metrics matter more with mixed
# numeric/one-hot data; default Minkowski/Euclidean used here)

os.environ["LOKY_MAX_CPU_COUNT"] = "1"

with joblib.parallel_backend("threading"):
    best_knn, best_k, best_f1 = None, None, -1
    for k in [3, 5, 7, 9, 11, 15]:
        pipe = Pipeline([('prep', preprocessor), ('model', KNeighborsClassifier(n_neighbors=k,n_jobs=1))])
        pipe.fit(X_train, y_train)
        f1 = f1_score(y_test, pipe.predict(X_test), average='weighted')
        if f1 > best_f1:
            best_f1, best_k, best_knn = f1, k, pipe
    print(f"Best k = {best_k} (weighted F1 = {best_f1:.4f})")
    evaluate('K-Nearest Neighbors', best_knn, X_train, y_train, X_test, y_test)

In [ ]:
# 3. Naive Bayes (Gaussian) — assumes conditional independence of features given the class;
# a strong assumption here since e.g. loan_amount and loan_to_income_ratio are correlated.
nb_pipe = Pipeline([('prep', preprocessor), ('model', GaussianNB())])
evaluate('Naive Bayes (Gaussian)', nb_pipe, X_train, y_train, X_test, y_test)


In [ ]:
# 4. Decision Tree Classifier (tune max_depth, visualise tree)
best_dt, best_depth, best_f1 = None, None, -1
for depth in [3, 5, 7, 10, None]:
    pipe = Pipeline([('prep', preprocessor), ('model', DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_STATE))])
    pipe.fit(X_train, y_train)
    f1 = f1_score(y_test, pipe.predict(X_test), average='weighted')
    if f1 > best_f1:
        best_f1, best_depth, best_dt = f1, depth, pipe
print(f"Best max_depth = {best_depth} (weighted F1 = {best_f1:.4f})")
evaluate('Decision Tree Classifier', best_dt, X_train, y_train, X_test, y_test)


In [ ]:
plt.figure(figsize=(20, 10))
plot_tree(best_dt.named_steps['model'], max_depth=3, feature_names=feature_names,
          class_names=[str(c) for c in sorted(y.unique())], filled=True, fontsize=8)
plt.title(f'Decision Tree (max_depth={best_depth}, shown to depth 3)')
plt.tight_layout()
plt.show()


In [ ]:
# 5. Support Vector Machine (SVC) — tune C and kernel; features already scaled in preprocessor
X_s, y_s = X_train.iloc[:2500], y_train.iloc[:2500]
best_svc, best_params, best_f1 = None, None, -1.0

for C in [0.5, 1.0, 5.0]:
    pipe = Pipeline([
        ('prep', preprocessor),
        ('model', SVC(C=C, kernel='rbf', max_iter=1000, random_state=RANDOM_STATE))
    ]).fit(X_s, y_s)
    f1 = f1_score(y_test, pipe.predict(X_test), average='weighted')
    if f1 > best_f1:
        best_f1, best_params, best_svc = f1, {'C': C, 'kernel': 'rbf'}, pipe

print(f"Best params (SVC) = {best_params} (weighted F1 = {best_f1:.4f})")
evaluate('Support Vector Machine', best_svc, X_s, y_s, X_test, y_test)

### D2. Evaluation — Confusion Matrices & Preliminary Comparison Table

In [ ]:
class_labels = sorted(y.unique())
fig, axes = plt.subplots(1, 5, figsize=(24, 5))
for ax, (name, r) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, r['preds'], labels=class_labels)
    disp = ConfusionMatrixDisplay(cm, display_labels=class_labels)
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
summary = pd.DataFrame({
    name: {'Accuracy': r['Accuracy'], 'Precision': r['Precision'],
           'Recall': r['Recall'], 'F1_weighted': r['F1_weighted']}
    for name, r in results.items()
}).T.sort_values('F1_weighted', ascending=False)

summary.style.background_gradient(cmap='Greens')


---
# Classification Track — Part B (Review 2)
Part B adds 5 more algorithms (Random Forest, AdaBoost, Gradient Boosting, Bagging, MLP) trained on the **same** `X_train / X_test` split and the **same** `preprocessor` as Part A, followed by ROC-AUC for all 10 models, a consolidated comparison table, cross-validation, hyperparameter tuning of the best model, and final model selection.

In [ ]:
from sklearn.ensemble import (RandomForestClassifier, AdaBoostClassifier,
                              GradientBoostingClassifier, BaggingClassifier)
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve, auc
from sklearn.preprocessing import label_binarize
from sklearn.base import clone

## Section A1 — Part B algorithms
Hyper-parameters are tuned with `GridSearchCV` on the **training set only** (3-fold, weighted F1), so the test set stays untouched until final evaluation.

In [ ]:
# Sample subset for fast Part B training (identical to SVC strategy in Part A)
X_s, y_s = X_train.iloc[:5000], y_train.iloc[:5000]

def tune_and_eval(name, base_model, param_grid, cv=3):
    """Grid-search a model on subset, then evaluate on full test set."""
    pipe = Pipeline([('prep', preprocessor), ('model', base_model)])
    gs = GridSearchCV(pipe, param_grid, scoring='f1_weighted',
                      cv=StratifiedKFold(cv, shuffle=True, random_state=RANDOM_STATE),
                      n_jobs=-1)
    gs.fit(X_s, y_s)
    print(f"Best params for {name}: {gs.best_params_}  (CV weighted F1 = {gs.best_score_:.4f})")
    evaluate(name, gs.best_estimator_, X_s, y_s, X_test, y_test)
    return gs

### 6. Random Forest Classifier — tune `n_estimators`, show feature importance

In [ ]:
rf_gs = tune_and_eval('Random Forest',
    RandomForestClassifier(random_state=RANDOM_STATE),
    {'model__n_estimators': [100, 200], 'model__max_depth': [12, 20]})

In [ ]:
rf_best = results['Random Forest']['model']
imp = pd.Series(rf_best.named_steps['model'].feature_importances_,
                index=rf_best.named_steps['prep'].get_feature_names_out()).sort_values(ascending=False).head(15)
plt.figure(figsize=(9, 6))
sns.barplot(x=imp.values, y=imp.index, color=sns.color_palette('colorblind')[0])
plt.title('Random Forest — Top 15 Feature Importances')
plt.xlabel('Importance'); plt.ylabel('Feature')
plt.tight_layout(); plt.show()

### 7. AdaBoost Classifier — tune `n_estimators` and `learning_rate`

In [ ]:
ada_gs = tune_and_eval('AdaBoost',
    AdaBoostClassifier(random_state=RANDOM_STATE),
    {'model__n_estimators': [50, 100], 'model__learning_rate': [0.05, 0.1, 0.5]})

### 8. Gradient Boosting Classifier (sklearn GBM)

In [ ]:
gb_gs = tune_and_eval('Gradient Boosting',
    GradientBoostingClassifier(random_state=RANDOM_STATE),
    {'model__n_estimators': [100, 200], 'model__learning_rate': [0.05, 0.1, 0.2], 'model__max_depth': [2, 3, 4]})

### 9. Bagging Classifier — Decision Tree as base estimator

In [ ]:
bag_gs = tune_and_eval('Bagging',
    BaggingClassifier(estimator=DecisionTreeClassifier(random_state=RANDOM_STATE), random_state=RANDOM_STATE, n_jobs=1),
    {'model__n_estimators': [10, 50, 100], 'model__max_samples': [0.5, 0.8, 1.0]})

### 10. MLP Classifier (Neural Network) — tune hidden layer sizes and activation

In [ ]:
mlp_gs = tune_and_eval('MLP Neural Network',
    MLPClassifier(max_iter=500, early_stopping=True, random_state=RANDOM_STATE),
    {'model__hidden_layer_sizes': [(32,), (64,), (64, 32), (128, 64)],
     'model__activation': ['relu', 'tanh']})

## Section A2 — ROC-AUC for all 10 algorithms
Part A did not report ROC-AUC, so it is computed here for every fitted model. SVC has no `predict_proba` by default, so its `decision_function` score is used (AUC only depends on ranking). Multi-class targets use One-vs-Rest.

In [ ]:
def get_scores(pipe, X):
    if hasattr(pipe, 'predict_proba'):
        return pipe.predict_proba(X)
    return pipe.decision_function(X)

def compute_auc(pipe, X, y_true):
    s = get_scores(pipe, X)
    classes = sorted(y_true.unique())
    if len(classes) == 2:
        return roc_auc_score(y_true, s[:, 1] if s.ndim == 2 else s)
    Y = label_binarize(y_true, classes=classes)          # One-vs-Rest for multi-class
    return roc_auc_score(Y, s, average='weighted')

for name, r in results.items():
    r['ROC_AUC'] = compute_auc(r['model'], X_test, y_test)
    print(f"{name:28s} ROC-AUC = {r['ROC_AUC']:.4f}")

## Section A2 — Consolidated 10-algorithm comparison table

In [ ]:
full_summary = pd.DataFrame({
    name: {'Accuracy': r['Accuracy'], 'Precision': r['Precision'], 'Recall': r['Recall'],
           'F1_weighted': r['F1_weighted'], 'ROC_AUC': r['ROC_AUC']}
    for name, r in results.items()
}).T.sort_values('F1_weighted', ascending=False)
full_summary.insert(0, 'Rank', range(1, len(full_summary) + 1))
assert len(full_summary) == 10, f"Expected 10 models, found {len(full_summary)}"
full_summary.style.format({c: '{:.4f}' for c in full_summary.columns if c != 'Rank'}).background_gradient(cmap='Greens', subset=['Accuracy','Precision','Recall','F1_weighted','ROC_AUC'])

In [ ]:
# Confusion matrices for all 10 models
fig, axes = plt.subplots(2, 5, figsize=(26, 10))
for ax, (name, r) in zip(axes.flatten(), results.items()):
    cm = confusion_matrix(y_test, r['preds'], labels=class_labels)
    ConfusionMatrixDisplay(cm, display_labels=class_labels).plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontsize=10)
plt.suptitle('Confusion Matrices — All 10 Classifiers', fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
# ROC curves (binary case); multi-class would need one curve per class
if len(class_labels) == 2:
    plt.figure(figsize=(9, 7))
    for name, r in results.items():
        s = get_scores(r['model'], X_test)
        s = s[:, 1] if s.ndim == 2 else s
        fpr, tpr, _ = roc_curve(y_test, s)
        plt.plot(fpr, tpr, label=f"{name} (AUC={auc(fpr, tpr):.3f})")
    plt.plot([0, 1], [0, 1], 'k--', label='Chance')
    plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
    plt.title('ROC Curves — All 10 Classifiers'); plt.legend(fontsize=8, loc='lower right')
    plt.tight_layout(); plt.show()

In [ ]:
# Bar chart of the main metrics
full_summary[['Accuracy', 'F1_weighted', 'ROC_AUC']].plot(kind='bar', figsize=(12, 5), colormap='tab10')
plt.title('Model Comparison — Accuracy, F1 (weighted), ROC-AUC')
plt.ylabel('Score'); plt.xlabel('Model'); plt.xticks(rotation=45, ha='right')
plt.ylim(max(0, full_summary[['Accuracy','F1_weighted','ROC_AUC']].min().min() - 0.05), 1.0)
plt.legend(loc='lower right'); plt.tight_layout(); plt.show()

## Section A3 — Final model selection
5-fold stratified cross-validation (on the training set) for the top-2 models, then hyperparameter tuning of the best one with before/after comparison.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
top2 = full_summary.index[:2].tolist()
cv_rows = {}
for name in top2:
    scores = cross_val_score(clone(results[name]['model']), X_s, y_s, cv=skf, scoring='f1_weighted', n_jobs=-1)
    cv_rows[name] = {'CV mean F1': scores.mean(), 'CV std': scores.std()}
pd.DataFrame(cv_rows).T

In [ ]:
best_name = full_summary.index[0]
print("Best model on test F1:", best_name)

# Search grids for further tuning (optimized for fast convergence)
final_grids = {
    'Random Forest':     {'model__n_estimators': [100, 200], 'model__max_depth': [15, 20],
                          'model__min_samples_split': [2, 5], 'model__max_features': ['sqrt']},
    'Gradient Boosting': {'model__n_estimators': [100, 200], 'model__learning_rate': [0.05, 0.1],
                          'model__max_depth': [3, 4], 'model__subsample': [0.8, 1.0]},
    'AdaBoost':          {'model__n_estimators': [50, 100, 150], 'model__learning_rate': [0.05, 0.1, 0.5]},
    'Bagging':           {'model__n_estimators': [30, 50], 'model__max_samples': [0.7, 1.0]},
    'MLP Neural Network':{'model__hidden_layer_sizes': [(64,), (64, 32)],
                          'model__activation': ['relu'], 'model__alpha': [1e-4, 1e-3]},
    'Logistic Regression':{'model__C': [0.01, 0.1, 1, 10, 100]},
    'K-Nearest Neighbors':{'model__n_neighbors': [5, 9, 15, 25], 'model__weights': ['uniform', 'distance'],
                           'model__metric': ['euclidean', 'manhattan']},
    'Decision Tree Classifier': {'model__max_depth': [3, 5, 7, 10, 15], 'model__min_samples_leaf': [1, 5, 10]},
    'Naive Bayes (Gaussian)': {'model__var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6]},
    'Support Vector Machine': {'model__C': [0.5, 1, 5, 10], 'model__kernel': ['rbf', 'linear']},
}
before = results[best_name].copy()

final_gs = GridSearchCV(clone(results[best_name]['model']), final_grids[best_name],
                        scoring='f1_weighted', cv=skf, n_jobs=-1)
final_gs.fit(X_s, y_s)
print("Best params:", final_gs.best_params_)
print(f"Best CV weighted F1: {final_gs.best_score_:.4f}")

In [ ]:
final_model = final_gs.best_estimator_
preds = final_model.predict(X_test)
after = {'Accuracy': accuracy_score(y_test, preds),
         'Precision': precision_score(y_test, preds, average='weighted', zero_division=0),
         'Recall': recall_score(y_test, preds, average='weighted', zero_division=0),
         'F1_weighted': f1_score(y_test, preds, average='weighted', zero_division=0),
         'ROC_AUC': compute_auc(final_model, X_test, y_test)}
comparison = pd.DataFrame({'Before tuning': {k: before[k] for k in after}, 'After tuning': after})
comparison['Improvement'] = comparison['After tuning'] - comparison['Before tuning']
comparison.style.format('{:.4f}')

In [ ]:
print(classification_report(y_test, preds))
ConfusionMatrixDisplay.from_predictions(y_test, preds, cmap='Blues')
plt.title(f'Final Model: {best_name} (tuned) — Confusion Matrix')
plt.tight_layout(); plt.show()

# Save the final model for the GUI / deployment bonus
os.makedirs('../models', exist_ok=True)
joblib.dump(final_model, '../models/final_classifier.pkl')
print("Saved ../models/final_classifier.pkl")